In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
# Read in the 2 csv files, need to combine them.
photometry = pd.read_csv('/data/zsl23/DustPedia_Aperture_Photometry_2.2.csv')
distance = pd.read_csv('/data/zsl23/DustPedia_HyperLEDA_Herschel.csv')

In [4]:
photometry.head()

,name,ra,dec,semimaj_arcsec,axial_ratio,pos_angle,global_flag,GALEX_FUV,GALEX_FUV_err,GALEX_FUV_flag,...,PACS_160_flag,SPIRE_250,SPIRE_250_err,SPIRE_250_flag,SPIRE_350,SPIRE_350_err,SPIRE_350_flag,SPIRE_500,SPIRE_500_err,SPIRE_500_flag
0,ESO149-013,0.69450,-52.77158,77.942286,1.000021,135.000038,NaN,NaN,NaN,NaN,...,NaN,0.049630,0.082991,NaN,0.007900,0.058638,NaN,0.007302,0.034583,NaN
1,NGC0007,2.08695,-29.91534,116.668752,1.527756,119.306955,NaN,0.002792,0.000131,NaN,...,NaN,0.419146,0.108799,NaN,0.343956,0.066856,NaN,0.168743,0.048714,NaN
2,ESO410-005,3.88020,-32.18061,86.990725,1.108053,-34.807567,NaN,0.000170,0.000015,NaN,...,NaN,0.049891,0.087069,NaN,0.015662,0.065673,NaN,0.009790,0.033119,NaN
3,IC0010,5.09625,59.29310,542.405824,1.185928,-2.622338,NaN,NaN,NaN,NaN,...,NaN,110.582865,59.084049,NaN,52.926953,40.419363,NaN,21.082392,14.078462,NaN
4,NGC0115,6.69330,-33.67698,129.879511,1.671339,36.707168,NaN,0.003101,0.000142,NaN,...,NaN,0.912792,0.129122,NaN,0.495309,0.097805,NaN,0.259551,0.060057,NaN


In [5]:
distance = distance.rename(columns={'dist_best':'distance'})
distance.head()

,objname,ra2000,de2000,t,t_err,type,logd25,d25,bt,incl,ned_v_helio,ned_v_corr,ned_dist_z_corr,ned_dist_0,hyperleda_v_helio,hyperleda_dist_z_helio,hyperleda_dist_0,distance
0,ESO149-013,0.69450,-52.77158,9.9,0.5,IB,1.07,1.174898,15.71,90.0,1500.0,1483.0,20.248498,NaN,1498,20.453304,NaN,20.248498
1,NGC0007,2.08695,-29.91534,4.8,0.7,Sc,1.39,2.454709,14.34,86.6,1495.0,1502.0,20.507919,20.62,1499,20.466958,20.606305,20.606305
2,ESO410-005,3.88020,-32.18061,0.4,2.3,S0-a,1.11,1.288250,14.93,51.8,159.0,164.0,2.239214,1.90,36,0.491535,1.940886,1.940886
3,IC0010,5.09625,59.29310,9.9,0.6,IB,1.82,6.606935,11.78,31.1,-348.0,-69.0,-0.942108,0.88,-345,NaN,0.794328,0.794328
4,NGC0115,6.69330,-33.67698,3.9,0.8,SBbc,1.32,2.089296,13.67,74.4,1825.0,1821.0,24.863463,29.98,1828,24.959039,28.707818,28.707818


In [6]:
# only want the distance column and objname for merging
bestdistance=distance[['objname','distance']]
bestdistance.head()

,objname,distance
0,ESO149-013,20.248498
1,NGC0007,20.606305
2,ESO410-005,1.940886
3,IC0010,0.794328
4,NGC0115,28.707818


In [7]:
combineddf = pd.merge(photometry, bestdistance, left_on='name', right_on='objname').drop(columns=['objname','ra','dec','semimaj_arcsec','axial_ratio','pos_angle'])
# insert redshift column filled with 0s
combineddf.insert(1,'redshift',0) #changes the df, but does not output the df, so don't put combineddf in front
# move the distance column next to redshift due to formatting requirements
cols = list(combineddf.columns)
cols = [cols[0]]+ [cols[1]]+ [cols[-1]] + cols[2:-1]
combineddf = combineddf[cols] 
combineddf.head()

,name,redshift,distance,global_flag,GALEX_FUV,GALEX_FUV_err,GALEX_FUV_flag,GALEX_NUV,GALEX_NUV_err,GALEX_NUV_flag,...,PACS_160_flag,SPIRE_250,SPIRE_250_err,SPIRE_250_flag,SPIRE_350,SPIRE_350_err,SPIRE_350_flag,SPIRE_500,SPIRE_500_err,SPIRE_500_flag
0,ESO149-013,0,20.248498,NaN,NaN,NaN,NaN,-6.090620e-07,0.000044,N,...,NaN,0.049630,0.082991,NaN,0.007900,0.058638,NaN,0.007302,0.034583,NaN
1,NGC0007,0,20.606305,NaN,0.002792,0.000131,NaN,3.131622e-03,0.000110,NaN,...,NaN,0.419146,0.108799,NaN,0.343956,0.066856,NaN,0.168743,0.048714,NaN
2,ESO410-005,0,1.940886,NaN,0.000170,0.000015,NaN,5.051966e-04,0.000029,NaN,...,NaN,0.049891,0.087069,NaN,0.015662,0.065673,NaN,0.009790,0.033119,NaN
3,IC0010,0,0.794328,NaN,NaN,NaN,NaN,8.263665e+01,332.637807,NaN,...,NaN,110.582865,59.084049,NaN,52.926953,40.419363,NaN,21.082392,14.078462,NaN
4,NGC0115,0,28.707818,NaN,0.003101,0.000142,NaN,3.916178e-03,0.000110,NaN,...,NaN,0.912792,0.129122,NaN,0.495309,0.097805,NaN,0.259551,0.060057,NaN


In [8]:
combineddf['global_flag'].unique()

array([nan, 'c', 'C'], dtype=object)

In [9]:
print('The total number of galaxies in this dataset is:',len(combineddf))

The total number of galaxies in this dataset is: 875


In [10]:
# filter out the global flagged galaxies
globalgaldf = combineddf[combineddf['global_flag'].isna()]
print('The total number of galaxies after filtering out the c and C flagged ones is:', len(globalgaldf))
# okay so this is what we want to work with.
globalgaldf = globalgaldf.drop(columns=['global_flag'])

The total number of galaxies after filtering out the c and C flagged ones is: 792


In [11]:
# replace those individual flagged filters with nan values 
# every column that ends in "_flag", paired with its value + err columns
filter_flags = {}
for col in globalgaldf.columns:
    if col.endswith('_flag'):
        base = col[:-len('_flag')] # this gives the base name of the filter
        filter_flags[col] = [base, f'{base}_err'] # create a mapping dictionary

# any flag containing A, C, N, E should be nulled
flag_letters = 'ACNE'

for flag_col, value_cols in filter_flags.items(): # .items() turn dictionary into tuples
    mask = globalgaldf[flag_col].str.contains(f'[{flag_letters}]', case=True, na=False, regex=True)
    # this also returns a boolean mask. So if the column is empty for a specific row, it returns False (set by na).
    globalgaldf.loc[mask, value_cols] = np.nan # [row, column(s)]
    globalgaldf[value_cols] = globalgaldf[value_cols]*1000 # convert each flux column and each error column from Jy to mJy

In [12]:
# just to see if theres any flag left behind
check = globalgaldf.to_csv('check_flags.csv')

In [13]:
# now we can drop the flag columns too
for col in globalgaldf.columns:
    if col.endswith('_flag'):
        globalgaldf = globalgaldf.drop(columns=col)

In [14]:
# some tedious renaming tasks to fit CIGALE syntax ...
rename_map = {
    "name": "#id",
    "GALEX_FUV": "galex.FUV", "GALEX_FUV_err": "galex.FUV_err",
    "GALEX_NUV": "galex.NUV", "GALEX_NUV_err": "galex.NUV_err",
    "SDSS_u": "sloan.sdss.u", "SDSS_u_err": "sloan.sdss.u_err",
    "SDSS_g": "sloan.sdss.g", "SDSS_g_err": "sloan.sdss.g_err",
    "SDSS_r": "sloan.sdss.r", "SDSS_r_err": "sloan.sdss.r_err",
    "SDSS_i": "sloan.sdss.i", "SDSS_i_err": "sloan.sdss.i_err",
    "SDSS_z": "sloan.sdss.z", "SDSS_z_err": "sloan.sdss.z_err",
    "2MASS_J": "2mass.J", "2MASS_J_err": "2mass.J_err",
    "2MASS_H": "2mass.H", "2MASS_H_err": "2mass.H_err",
    "2MASS_Ks": "2mass.Ks", "2MASS_Ks_err": "2mass.Ks_err",
    "WISE_3.4": "wise.W1", "WISE_3.4_err": "wise.W1_err",
    "WISE_4.6": "wise.W2", "WISE_4.6_err": "wise.W2_err",
    "WISE_12": "wise.W3", "WISE_12_err": "wise.W3_err",
    "WISE_22": "wise.W4", "WISE_22_err": "wise.W4_err",
    "Spitzer_3.6": "spitzer.irac.I1", "Spitzer_3.6_err": "spitzer.irac.I1_err",
    "Spitzer_4.5": "spitzer.irac.I2", "Spitzer_4.5_err": "spitzer.irac.I2_err",
    "Spitzer_5.8": "spitzer.irac.I3", "Spitzer_5.8_err": "spitzer.irac.I3_err",
    "Spitzer_8.0": "spitzer.irac.I4", "Spitzer_8.0_err": "spitzer.irac.I4_err",
    "Spitzer_24": "spitzer.mips.24mu", "Spitzer_24_err": "spitzer.mips.24mu_err",
    "Spitzer_70": "spitzer.mips.70mu", "Spitzer_70_err": "spitzer.mips.70mu_err",
    "Spitzer_160": "spitzer.mips.160mu", "Spitzer_160_err": "spitzer.mips.160mu_err",
    "PACS_70": "herschel.pacs.blue", "PACS_70_err": "herschel.pacs.blue_err",
    "PACS_100": "herschel.pacs.green", "PACS_100_err": "herschel.pacs.green_err",
    "PACS_160": "herschel.pacs.red", "PACS_160_err": "herschel.pacs.red_err",
    "SPIRE_250": "herschel.spire.PSW", "SPIRE_250_err": "herschel.spire.PSW_err",
    "SPIRE_350": "herschel.spire.PMW", "SPIRE_350_err": "herschel.spire.PMW_err",
    "SPIRE_500": "herschel.spire.PLW", "SPIRE_500_err": "herschel.spire.PLW_err",
}

globalgaldf = globalgaldf.rename(columns=rename_map)
globalgaldf.to_csv('basedf.csv', index=False)

In [15]:
globalgaldf.head()

,#id,redshift,distance,galex.FUV,galex.FUV_err,galex.NUV,galex.NUV_err,sloan.sdss.u,sloan.sdss.u_err,sloan.sdss.g,...,herschel.pacs.green,herschel.pacs.green_err,herschel.pacs.red,herschel.pacs.red_err,herschel.spire.PSW,herschel.spire.PSW_err,herschel.spire.PMW,herschel.spire.PMW_err,herschel.spire.PLW,herschel.spire.PLW_err
0,ESO149-013,0,20.248498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,49.629813,82.990714,7.899533,58.638459,7.301990,34.582998
1,NGC0007,0,20.606305,2.792007,0.130765,3.131622,0.109649,NaN,NaN,NaN,...,-585.993205,550.268549,479.281338,561.583315,419.146475,108.798696,343.955863,66.855994,168.743455,48.714171
2,ESO410-005,0,1.940886,0.169843,0.015266,0.505197,0.029494,NaN,NaN,NaN,...,606.719823,298.355089,575.689527,351.670292,49.891475,87.069055,15.661851,65.672827,9.789736,33.119280
3,IC0010,0,0.794328,NaN,NaN,82636.651201,332637.806656,NaN,NaN,NaN,...,216045.743398,37151.678713,214645.823529,118767.051682,110582.864955,59084.049307,52926.952540,40419.362730,21082.391740,14078.461687
4,NGC0115,0,28.707818,3.101272,0.142196,3.916178,0.110150,NaN,NaN,NaN,...,896.019859,600.566138,2157.135283,488.568231,912.792408,129.122311,495.308969,97.804529,259.551036,60.057107


In [16]:
allfiltergal = globalgaldf.dropna()
print('Only',len(allfiltergal),'galaxies have all', int(len(globalgaldf.columns[3:])/2),'bands!')
allfiltergal.to_csv('allfiltergal.txt', sep=' ', index=False)

Only 7 galaxies have all 27 bands!


In [17]:
# to see how many galaxies have complete bands without spitzer, since these columns are mostly empty
no_spitzer_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'spitzer' in f])
nospitzergal = no_spitzer_df.dropna()
nospitzergal.to_csv('nospitzergal.txt', sep=' ', index=False)
print(len(nospitzergal), 'galaxies have complete bands without Spitzer!')

78 galaxies have complete bands without Spitzer!


In [18]:
# to see how many galaxies have complete bands without sdss bands
no_sdss_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'sdss' in f])
nosdssgal = no_sdss_df.dropna()
nosdssgal.to_csv('nosdssgal.txt', sep=' ', index=False)
print(len(nosdssgal), 'galaxies have complete bands without SDSS!')

7 galaxies have complete bands without SDSS!


In [19]:
# to see how many galaxies have complete bands without spitzer.mips bands
no_spitzermips_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'spitzer.mips' in f])
nospitzermipsgal = no_spitzermips_df.dropna()
nospitzermipsgal.to_csv('nospitzermipsgal.txt', sep=' ', index=False)
print(len(nospitzermipsgal), 'galaxies have complete bands without spitzer.mips!')

30 galaxies have complete bands without spitzer.mips!


In [20]:
# to see how many galaxies have complete bands without sdss AND spitzer bands
no_sdss_spitzer_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'sdss' in f or 'spitzer' in f])
nosdssspitzergal = no_sdss_spitzer_df.dropna()
nosdssspitzergal.to_csv('nosdssspitzergal.txt', sep=' ', index=False)
print(len(nosdssspitzergal), 'galaxies have complete bands without SDSS and Spitzer!')

108 galaxies have complete bands without SDSS and Spitzer!


In [21]:
# to see how many galaxies have complete bands without PACS bands
no_pacs_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'pacs' in f])
nopacsgal = no_pacs_df.dropna()
nopacsgal.to_csv('nopacsgal.txt', sep=' ', index=False)
print(len(nopacsgal), 'galaxies have complete bands without PACS!')

38 galaxies have complete bands without PACS!


In [22]:
# to see how many galaxies have complete bands without PACS AND spitzer bands
no_pacs_spitzer_df = globalgaldf.drop(columns=[f for f in globalgaldf.columns if 'pacs' in f or 'spitzer' in f])
nopacsspitzergal = no_pacs_spitzer_df.dropna()
nopacsspitzergal.to_csv('nopacsspitzergal.txt', sep=' ', index=False)
print(len(nopacsspitzergal), 'galaxies have complete bands without PACS and Spitzer!')

486 galaxies have complete bands without PACS and Spitzer!
